In [10]:
%%writefile /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/CARMA_examples/code/CARMA_1KG.R
library(arrow)
library(data.table)
library(dplyr)
library(tibble)
library(tidyr)
library(ggplot2)
library(tidyverse)
library(plink2R)
library(MASS)
library(crayon)
library(pecotmr)
library(vroom)
library(pgenlibr)
library(Rfast)
library(mixsqp)

kriging_fix <- function(z, LD, n, z_thres = 2, logLR_thres = 2) {
  krig <- kriging_rss(z = z, R = LD, n = n)
  krig_outlier <- krig$conditional_dist %>%
    mutate(index = row_number()) %>%
    filter(abs(z) > z_thres & logLR > logLR_thres)
  z_fix <- z
  z_fix[krig_outlier$index] <- -z_fix[krig_outlier$index]
  list(z = z_fix,
       outlier_index  = krig_outlier$index,
       outlier_number = length(krig_outlier$index),
       output         = krig$conditional_dist)
}

# pecotmr GitHub version (conda not yet updated)
for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/pecotmr/R/",
                     full.names = TRUE, pattern = "\\.R$")) source(f)

# pecotmr GitHub version (conda not yet updated)
for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/",
                     full.names = TRUE, pattern = "\\.R$")) source(f)
select <- dplyr::select

setwd("/sc/arion/projects/ad-omics/haochen/rss_qc/CARMA/CARMA_2023_AD")
options(stringsAsFactors = FALSE)
options(arrow.unsafe_metadata = TRUE)

analysis_date <- "20260812"
per_gene_dir <- "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/CARMA_examples/result_KG/"
dir.create(per_gene_dir, showWarnings = FALSE, recursive = TRUE)

L_default <- 40
L_greedy_default <- 10
max_iter_default <- 50
phi <- 111326 / (111326 + 677663)
var_y_default <- 1 / (phi * (1 - phi))

select <- dplyr::select

ref_table <- read.csv("ref_table.csv") %>%
  mutate(base = locus.name)

ld_files <- tibble(
  base = sub("\\.parquet$", "", list.files(pattern = "\\.parquet$")),
  ld_file = list.files(pattern = "\\.parquet$")
)

ss_files <- tibble(
  base = sub("\\.processed$", "", list.files(pattern = "\\.processed$")),
  ss_file = list.files(pattern = "\\.processed$")
)

region_files <- inner_join(ld_files, ss_files, by = "base") %>%
  left_join(ref_table, by = "base") %>%
  arrange(chr, region_start, region_end, base)

missing_from_data <- anti_join(ref_table, region_files, by = "base") %>%
  select(locus.name, locus, chr, region_start, region_end)

cat("Matched", nrow(region_files), "regions with", 
    sum(file.exists(region_files$ld_file)), "LD files and",
    sum(file.exists(region_files$ss_file)), "summary-stat files.\n")
cat("Total variants across matched regions:",
    sum(vapply(region_files$ss_file, function(f) nrow(fread(f, sep = " ", header = TRUE, showProgress = FALSE)), integer(1))),
    "\n")

region_files %>% select(base, locus, chr, region_start, region_end, ld_file, ss_file)
missing_from_data

read_region_inputs <- function(row) {
  ld <- as.matrix(read_parquet(row$ld_file))
  ss <- fread(row$ss_file, sep = " ", header = TRUE)

  if (nrow(ld) != nrow(ss) || ncol(ld) != nrow(ss)) {
    stop("LD dimension does not match summary-stat rows for ", row$base,
         ": R = ", nrow(ld), " x ", ncol(ld), ", ss rows = ", nrow(ss))
  }

  list(ld = ld, ss = ss)
}

for (i in seq_len(nrow(region_files))) {
  row     <- region_files[i, , drop = FALSE]
  out_rds <- file.path(per_gene_dir, sprintf("region_%03d_%s.susie.rds", i, row$base))

  if (file.exists(out_rds)) { cat(sprintf("[SKIP] %d %s\n", i, row$base)); next }

  tryCatch({
    dat <- read_region_inputs(row)
    LD  <- dat$ld
    n   <- median(dat$ss$Neff, na.rm = TRUE)

    ## ---- kriging QC: flip outlying signs before fine-mapping ----
    z    <- dat$ss$BETA / dat$ss$SE
    krig <- kriging_fix(z = z, LD = LD, n = n)
    bhat <- dat$ss$BETA
    bhat[krig$outlier_index] <- -bhat[krig$outlier_index]   # sign flip; SE unchanged
    shat <- dat$ss$SE

    ## ---- fits on the kriging-corrected summary statistics ----
    fit_eb  <- susie_rss(bhat = bhat, shat = shat, R = LD, n = n,
                                              R_finite = 10000, 
                         R_mismatch = "eb_mix", var_y = var_y_default)
    fit_rss <- susie_rss(bhat = bhat, shat = shat, R = LD, n = n,
                         var_y = var_y_default)

    saveRDS(list(
      index   = i,
      base    = row$base,
      row     = row,
      n       = n,
      ss      = dat$ss,                       # original (unflipped) sumstats
      kriging_outlier_number = krig$outlier_number,
      kriging_outlier_index  = krig$outlier_index,
      kriging_output         = krig$output,
      fit_eb  = fit_eb,
      fit_rss = fit_rss
    ), out_rds)

    cat(sprintf("[OK]   %d %s  flipped=%d  eb_cs=%d rss_cs=%d\n",
                i, row$base, krig$outlier_number,
                length(fit_eb$sets$cs), length(fit_rss$sets$cs)))
  }, error = function(e) {
    cat(sprintf("[FAIL] %d %s: %s\n", i, row$base, conditionMessage(e)))
  })
}

Overwriting /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/CARMA_examples/code/CARMA_1KG.R


In [11]:
%%writefile /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/CARMA_examples/code/CARMA_1KG.sh
#!/bin/bash -l
#BSUB -J CARMA
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -n 2
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=50000]
#BSUB -W 28:00
#BSUB -o /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/CARMA_examples/log/CARMA_%J.out
#BSUB -e /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/CARMA_examples/log/CARMA_%J.err

export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin
workdir=/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/CARMA_examples/
mkdir -p ${workdir}/log

code_dir="$workdir/code"
tmp_root="$workdir/tmp"
tmp_dir="$tmp_root/${LSB_JOBID}"
mkdir -p "$tmp_dir"

export TMPDIR="$tmp_dir"
export TMP="$tmp_dir"
export TEMP="$tmp_dir"

# clean up scratch on exit
trap 'rm -rf "$tmp_dir"' EXIT

Rscript "$code_dir/CARMA_1KG.R"

Overwriting /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/CARMA_examples/code/CARMA_1KG.sh


## Fine-map the CARMA loci

In [7]:
%%writefile /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/CARMA_examples/code/CARMA_ADSP.R
library(arrow)
library(data.table)
library(dplyr)
library(tibble)
library(tidyr)
library(ggplot2)
library(tidyverse)
library(plink2R)
library(MASS)
library(crayon)
library(pecotmr)
library(rtracklayer)
library(GenomicRanges)
library(susieR)
library(BEDMatrix)
library(vroom)
library(mixsqp)
library(pgenlibr)
library(matrixStats)
library(Rfast)

# pecotmr GitHub version (conda not yet updated)
for (f in list.files(
  "/sc/arion/projects/ad-omics/haochen/rss_qc/pecotmr/R/",
  full.names = TRUE,
  pattern = "\\.R$"
)) {
  source(f)
}

# susieR GitHub version (conda not yet updated)
for (f in list.files(
  "/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/",
  full.names = TRUE,
  pattern = "\\.R$"
)) {
  source(f)
}

select <- dplyr::select
rename <- dplyr::rename

setwd("/hpc/users/sunh17/haochen/rss_qc/CARMA/CARMA_2023_AD")

options(
  stringsAsFactors = FALSE,
  arrow.unsafe_metadata = TRUE
)

analysis_date <- "20260812"


# ---- helpers ----
sketch_harmonize <- function(target_data, ref_variants, sketch) {
  qc_data <- allele_qc(match_min_prop = 0,
                       target_data    = target_data,
                       ref_variants   = ref_variants)
  qced <- qc_data$target_data_qced
  idx  <- match(qced$variant_id, colnames(sketch))
  if (anyNA(idx)) stop("Some qced variant_id values not found in sketch columns.")
  sketch2 <- sketch[, idx, drop = FALSE]
  flip_index <- which(qced$variants_id_original != qced$variant_id)
  if (length(flip_index) > 0)
    sketch2[, flip_index] <- -sketch2[, flip_index, drop = FALSE]
  colnames(sketch2) <- qced$variants_id_original
  list(sumstat = qced, sketch = sketch2,
       flipped_prop = length(flip_index) / ncol(sketch2))
}

kriging_fix <- function(z, LD, n, z_thres = 2, logLR_thres = 2) {
  krig <- kriging_rss(z = z, R = LD, n = n)
  krig_outlier <- krig$conditional_dist %>%
    mutate(index = row_number()) %>%
    filter(abs(z) > z_thres & logLR > logLR_thres)
  z_fix <- z
  z_fix[krig_outlier$index] <- -z_fix[krig_outlier$index]
  list(z = z_fix,
       outlier_index  = krig_outlier$index,
       outlier_number = length(krig_outlier$index),
       output         = krig$conditional_dist)
}


setwd("/sc/arion/projects/ad-omics/haochen/rss_qc/CARMA/CARMA_2023_AD")
options(stringsAsFactors = FALSE)
options(arrow.unsafe_metadata = TRUE)

per_gene_dir <- paste0(
  "/sc/arion/projects/ad-omics/haochen/rss_qc/",
  "2026_output/CARMA_examples/result_ADSP/"
)


dir.create(per_gene_dir, showWarnings = FALSE, recursive = TRUE)
ld_meta_file <- "/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/rss_ld_sketch/ld_meta_file.tsv"

L_default <- 40
L_greedy_default <- 10
max_iter_default <- 50
phi <- 111326 / (111326 + 677663)
var_y_default <- 1 / (phi * (1 - phi))

select <- dplyr::select

ref_table <- read.csv("ref_table.csv") %>%
  mutate(base = locus.name)

ld_files <- tibble(
  base = sub("\\.parquet$", "", list.files(pattern = "\\.parquet$")),
  ld_file = list.files(pattern = "\\.parquet$")
)

ss_files <- tibble(
  base = sub("\\.processed$", "", list.files(pattern = "\\.processed$")),
  ss_file = list.files(pattern = "\\.processed$")
)

region_files <- inner_join(ld_files, ss_files, by = "base") %>%
  left_join(ref_table, by = "base") %>%
  arrange(chr, region_start, region_end, base)

missing_from_data <- anti_join(ref_table, region_files, by = "base") %>%
  select(locus.name, locus, chr, region_start, region_end)

cat("Matched", nrow(region_files), "regions with", 
    sum(file.exists(region_files$ld_file)), "LD files and",
    sum(file.exists(region_files$ss_file)), "summary-stat files.\n")
cat("Total variants across matched regions:",
    sum(vapply(region_files$ss_file, function(f) nrow(fread(f, sep = " ", header = TRUE, showProgress = FALSE)), integer(1))),
    "\n")

region_files %>% select(base, locus, chr, region_start, region_end, ld_file, ss_file)
missing_from_data

read_region_inputs <- function(row) {
  ld <- as.matrix(read_parquet(row$ld_file))
  ss <- fread(row$ss_file, sep = " ", header = TRUE)

  if (nrow(ld) != nrow(ss) || ncol(ld) != nrow(ss)) {
    stop("LD dimension does not match summary-stat rows for ", row$base,
         ": R = ", nrow(ld), " x ", ncol(ld), ", ss rows = ", nrow(ss))
  }

  list(ld = ld, ss = ss)
}

for (i in seq_len(nrow(region_files))) {
row <- region_files[i, , drop = FALSE]
dat <- read_region_inputs(row)

sumstat <- dat$ss
chrom   <- unique(sumstat$CHR)
stopifnot(length(chrom) == 1)

## ---- liftOver hg19 -> hg38 ----
ch  <- import.chain("hg19ToHg38.over.chain")
gr  <- GRanges(seqnames = paste0("chr", sumstat$CHR),
               ranges   = IRanges(start = sumstat$BP, width = 1),
               row_id   = seq_len(nrow(sumstat)))
lifted  <- unlist(liftOver(gr, ch))
lift_dt <- data.table(row_id   = lifted$row_id,
                      CHR_hg38 = as.integer(sub("chr", "", as.character(seqnames(lifted)))),
                      BP_hg38  = start(lifted))

sumstat[, row_id := .I]
sumstat_hg38 <- merge(sumstat, lift_dt, by = "row_id")
sumstat_hg38 <- sumstat_hg38[CHR_hg38 == CHR]            # drop cross-chromosome lifts
n_dropped    <- nrow(sumstat) - nrow(sumstat_hg38)
cat(sprintf("Lifted %d / %d  (%d dropped)\n",
            nrow(sumstat_hg38), nrow(sumstat), n_dropped))

## region must be in hg38 coordinates (the sketch panel is hg38)
sumstat_hg38 <- sumstat_hg38 %>% arrange(BP_hg38)
start  <- min(sumstat_hg38$BP_hg38)
end    <- max(sumstat_hg38$BP_hg38)
region <- paste0("chr", chrom, ":", start, "-", end)
print(paste0("Region:", region))

## ---- load hg38 sketch reference ----
res_sketch           <- load_LD_matrix(ld_meta_file, region, return_genotype = TRUE)
res_sketch$LD_matrix <- scale(res_sketch$LD_matrix, center = TRUE, scale = TRUE)
sketch_ref           <- res_sketch$ref_panel

## ---- harmonize to hg38 positions ----
sumstat_hg38 <- sumstat_hg38 %>%
  dplyr::select(chrom = CHR_hg38, pos = BP_hg38, A1, A2, z = Z, Neff = Neff, se = SE)

harm <- sketch_harmonize(target_data  = sumstat_hg38,
                         ref_variants = sketch_ref %>% select(-variant_id),
                         sketch       = res_sketch$LD_matrix)

n <- median(sumstat_hg38$Neff, na.rm = TRUE)            # <- fixed

## ---- kriging outlier fix ----
sketch_LD <- crossprod(harm$sketch) / (nrow(harm$sketch) - 1)
krig      <- kriging_fix(z = harm$sumstat$z, LD = sketch_LD, n = n)
harm$sumstat$z <- krig$z

## ---- fits (renamed so we don't clobber susie_rss()) ----
z      <- harm$sumstat$z
print("==z==")
print(z[1:10])
bhat = z * harm$sumstat$se
print("==bhat==")
print(bhat[1:10])
se = harm$sumstat$se
print("==SE==")
print(se[1:10])

sketch = harm$sketch
fit_eb  <- susie_rss(bhat = bhat, shat = se, X = harm$sketch, n = n,
                     R_finite = 10000, 
                     R_mismatch = "eb_mix", var_y = var_y_default)

fit_rss <- susie_rss(bhat = bhat, shat = se, X = harm$sketch, n = n,
                    var_y = var_y_default)


## ---- save everything needed ----
output <- list(
  index            = i,
  base             = row$base,
  row              = row,
  region           = region,
  chrom            = chrom,
  n_sample         = n,
  n_lift_dropped   = n_dropped,
  flipped_prop     = harm$flipped_prop,
  kriging_outlier_number = krig$outlier_number,
  kriging_outlier_index  = krig$outlier_index,
  kriging_output         = krig$output,
  variant_id       = colnames(sketch),
  z                = z,
  fit_eb           = fit_eb,
  fit_rss          = fit_rss
)

out_path <- file.path(per_gene_dir, sprintf("region_%03d_%s.rds", i, row$base))
saveRDS(output, out_path)
cat(sprintf("[OK] idx %d %s  eb_cs=%d rss_cs=%d krig_outliers=%d\n",
            i, row$base, length(fit_eb$sets$cs), length(fit_rss$sets$cs),
            krig$outlier_number))

}

Overwriting /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/CARMA_examples/code/CARMA_ADSP.R


In [5]:
%%writefile /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/CARMA_examples/code/CARMA_ADSP.sh
#!/bin/bash -l
#BSUB -J CARMA
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -n 2
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=50000]
#BSUB -W 28:00
#BSUB -o /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/CARMA_examples/log/CARMA_%J.out
#BSUB -e /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/CARMA_examples/log/CARMA_%J.err

export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin
workdir=/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/CARMA_examples/
mkdir -p ${workdir}/log

code_dir="$workdir/code"
tmp_root="$workdir/tmp"
tmp_dir="$tmp_root/${LSB_JOBID}"
mkdir -p "$tmp_dir"

export TMPDIR="$tmp_dir"
export TMP="$tmp_dir"
export TEMP="$tmp_dir"

# clean up scratch on exit
trap 'rm -rf "$tmp_dir"' EXIT

Rscript "$code_dir/CARMA_ADSP.R"

Overwriting /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/CARMA_examples/code/CARMA_ADSP.sh
